# 05. MobileNetV3: улучшение классификатора и калибровка POI

Эксперимент сравнивает gallery cosine и сбалансированный Ridge на исходных MobileNetV3-Small эмбеддингах.
Линейный слой обучается только на positive-train. Alpha и per-POI score/margin пороги выбираются на val/negative-val.
Test не влияет на выбор. Предыдущий CNN fine-tuning не улучшил validation-верификацию и не был принят.
Достижение accuracy около 85% не равнозначно true accept 85%.


In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
ROOT=Path.cwd().resolve()
if not (ROOT/'config/mobilenetv3_calibrated.json').exists():ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.poi_metric import export_calibrated,CalibratedPOIVerifier
from src.poi_baseline import sha256_file
OUTPUT=ROOT/'models/mobilenetv3_calibrated'
if not OUTPUT.exists():export_calibrated(ROOT,OUTPUT)
metadata=json.loads((OUTPUT/'metadata.json').read_text())
for name,digest in metadata['artifact_sha256'].items():assert sha256_file(OUTPUT/name)==digest,name
for name,digest in metadata['input_split_sha256'].items():assert sha256_file(ROOT/f'data/annotations/{name}.csv')==digest,name
metrics=json.loads((OUTPUT/'metrics.json').read_text())
display(pd.Series(metadata))


I:\Невидаль\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


schema_version                                                                    3
model_type                                                   mobilenetv3_calibrated
architecture                                                     mobilenet_v3_small
poi_ids                           [moscow/arbat/arseny_morozov_house, moscow/arb...
scorer                                                               balanced_ridge
ridge_alpha                                                                     0.1
backbone_training_performed                                                   False
linear_head_training_performed                                                 True
training_split                                                  positive_train_only
calibration_split                                     positive_val_and_negative_val
test_used_for_selection                                                       False
seed                                                                        

## 1. Выбор только по validation

Классы имеют равный суммарный вес при обучении Ridge, число фотографий в большом классе не доминирует.
Ответ регрессии может выходить за [0,1]; он не объявляется вероятностью.
Каждому POI подбираются два порога. Динамическое программирование распределяет **общий**, а не отдельный
для каждого класса бюджет false accepts на negative-val и ошибочных известных целях.
В малых классах такая калибровка может переучитывать validation; test нужен для выявления этого риска.


In [2]:
display(pd.read_csv(OUTPUT/'validation_candidates.csv'))
display(pd.Series(json.loads((OUTPUT/'selection.json').read_text())))
thresholds=json.loads((OUTPUT/'thresholds_by_poi.json').read_text())
display(pd.DataFrame(thresholds).T)


,scorer,ridge_alpha,accuracy,macro_f1,true_accept_rate,external_any_poi_far,hard_wrong_far
0,gallery_cosine,NaN,0.788079,0.733041,0.615894,0.008726,0.013245
1,balanced_ridge,0.01,0.801325,0.771078,0.576159,0.008726,0.006623
2,balanced_ridge,0.10,0.854305,0.841510,0.629139,0.008726,0.000000
3,balanced_ridge,1.00,0.827815,0.764385,0.629139,0.008726,0.000000


selected           {'scorer': 'balanced_ridge', 'ridge_alpha': 0....
selection_split                                 val_and_negative_val
test_used                                                      False
dtype: object

,score_threshold,margin_threshold
moscow/arbat/arseny_morozov_house,0.541465,0.360063
moscow/arbat/pashkov_house,0.542011,0.389821
moscow/khamovniki/khrushchev_seleznev_house,0.588738,0.365030
moscow/khamovniki/pushkin_museum,0.444772,0.219713
moscow/ostankinskiy/armenia,0.718193,0.554873
moscow/ostankinskiy/central_pavilion,0.489123,0.257163
moscow/ostankinskiy/cosmos,0.986799,0.760151
moscow/ostankinskiy/main_entrance,0.436339,0.236555
moscow/ostankinskiy/moscow_sun,1.032168,0.911614
moscow/ostankinskiy/ostankino_tower,0.298353,0.171709


## 2. Test: классификация и отказ оцениваются отдельно

Accuracy и macro-F1 описывают выбор одного из 23 POI без отказа.
True accept учитывает пороги для правильной цели.
External any-POI FAR считает negative-фото ошибкой, если принято хотя бы за один целевой POI.
Превышение FAR не исправляется подбором по test и не маскируется средней ошибкой по всем парам.


In [3]:
display(pd.DataFrame([{ 'split':name,'accuracy':metrics[name]['accuracy'],'macro_f1':metrics[name]['macro_f1'],
 'true_accept':metrics[name]['true_accept_rate'],'external_any_poi_far':metrics[f'negative_{name}']['any_poi_far']}
 for name in ['val','test']]))
display(pd.read_csv(OUTPUT/'test_per_poi.csv'))
display(pd.read_csv(OUTPUT/'negative_test_groups.csv'))
print('Статус выпуска:', metadata['deployment_status'])


,split,accuracy,macro_f1,true_accept,external_any_poi_far
0,val,0.854305,0.841510,0.629139,0.008726
1,test,0.849315,0.846279,0.458904,0.050611


,poi_id,query_count,top1_accuracy,true_accept_rate,known_wrong_far,true_accepts,wrong_accepts
0,moscow/arbat/arseny_morozov_house,7,1.000000,0.571429,0.0,4,0
1,moscow/arbat/pashkov_house,7,1.000000,0.428571,0.0,3,0
2,moscow/khamovniki/khrushchev_seleznev_house,5,0.600000,0.000000,0.0,0,0
3,moscow/khamovniki/pushkin_museum,6,0.666667,0.500000,0.0,3,0
4,moscow/ostankinskiy/armenia,6,0.833333,0.000000,0.0,0,0
5,moscow/ostankinskiy/central_pavilion,6,0.666667,0.500000,0.0,3,0
6,moscow/ostankinskiy/cosmos,7,1.000000,0.000000,0.0,0,0
7,moscow/ostankinskiy/main_entrance,7,1.000000,0.714286,0.0,5,0
8,moscow/ostankinskiy/moscow_sun,8,0.875000,0.125000,0.0,1,0
9,moscow/ostankinskiy/ostankino_tower,10,0.900000,0.900000,0.0,9,0


,group_by,group,images,false_accepts,any_poi_far
0,negative_type,similar_arch,48,3,0.062500
1,negative_type,similar_classical_building,100,3,0.030000
2,negative_type,similar_decorated_building,50,2,0.040000
3,negative_type,similar_hotel,98,4,0.040816
4,negative_type,similar_mansion,50,1,0.020000
5,negative_type,similar_pavilion,177,5,0.028249
6,negative_type,similar_tower,50,11,0.220000
7,source_poi_id,moscow/aeroport/petrovsky_palace,50,1,0.020000
8,source_poi_id,moscow/donskoy/shukhov_tower,50,11,0.220000
9,source_poi_id,moscow/dorogomilovo/hotel_ukraina,50,4,0.080000


Статус выпуска: not_promoted_far_exceeded


## 3. Локальная проверка цели

При Ridge решении галерея используется для ближайшего эталона и происхождения; итоговый score принадлежит
обученному линейному слою. Это отличается от исходного max-cosine baseline и зафиксировано в metadata.
Автоматического переключения используемой версии нет.


In [4]:
verifier=CalibratedPOIVerifier(OUTPUT)
test=pd.read_csv(OUTPUT/'splits/test.csv',keep_default_na=False).sort_values('image_id')
example=test.iloc[0]
display(pd.Series(verifier.verify(ROOT/example.file_path,example.poi_id)))


accepted                                                              True
reason                                                        poi_verified
target_poi_id                moscow/ostankinskiy/friendship_of_the_peoples
predicted_poi_id             moscow/ostankinskiy/friendship_of_the_peoples
score                                                             0.816608
margin                                                             0.64348
score_type                         balanced_ridge_response_not_probability
score_is_probability                                                 False
reference_image_id                                 manual_64f68fdd6d8f76ae
reference_source                                                    manual
reference_author                                             Shilkova A.M.
reference_licence                                            Public Domain
reference_version                                64282c8db70a-37bbbca65dee
threshold_version        

## 4. Вывод

Классификация может улучшиться одновременно с ухудшением отказа на посторонних POI.
Кандидат, нарушивший ограничение FAR, не рекомендуется как замена строгой политики.
Для повышения true accept при низком FAR нужны больше независимых данных и проверка на новом закрытом полевом test.
Источник, автор и лицензия всех снимков сохранены в snapshots и галерее.


In [5]:
display(Markdown((ROOT/'reports/mobilenetv3_calibrated/summary.md').read_text(encoding='utf-8')))


# MobileNetV3: улучшение классификатора и политики отказа

Выбран scorer=balanced_ridge, ridge_alpha=0.1. Backbone ImageNet заморожен; обучаемый линейный слой использует только positive-train.
Сбалансированная Ridge-регрессия к one-hot меткам, с intercept. Отклик не является вероятностью и не обязан лежать в [0,1].
Отдельные score/margin-пороги каждого POI выбраны совместно динамическим программированием на positive-val и negative-val.
Общие validation-бюджеты: external any-POI FAR ≤1%, hard-wrong FAR ≤5%, known-wrong FAR ≤1%. Цель — максимум true accept, затем macro-F1 и accuracy.
Принимается только единственный лидирующий POI после прохождения его двух порогов. Галерея сохранена как контроль и источник ближайших эталонов; при scorer=Ridge решение принимает обученный слой.

| Test | Accuracy | Macro-F1 | True accept | External any-POI FAR |
|---|---:|---:|---:|---:|
| Frozen Small + global thresholds | 74.66% | 73.26% | 25.34% | 0.87% |
| Выбранный вариант | 84.93% | 84.63% | 45.89% | 5.06% |

## Наблюдение → интерпретация → действие
- Верных подтверждений 67/146 → true accept 45.89% → цель 85% не достигнута.
- External false accepts 29/573 → FAR 5.06% → validation-ограничение не гарантирует test FAR; не корректировать пороги по этому test.
- Требование FAR ≤1% на test НЕ выполнено → кандидат не рекомендуется как замена строгой политики → предыдущая версия остаётся доступной.
- Per-POI пороги дают больше гибкости, но на некоторые классы приходится 1–2 val-фото → высокий риск переучёта калибровки → считать выпуск экспериментальным и добрать независимые положительные/отрицательные кадры по классам.
- CNN fine-tuning с новым preprocessing не улучшил validation true accept → этот вариант не заменяет baseline → история отдельного эксперимента сохранена в models/mobilenetv3_finetuned.
- Гиперпараметры и пороги выбраны только на val/negative-val → test не участвовал в выборе текущего выпуска → итоговую проверку провести также на новом закрытом полевом наборе.
- Атрибуция source/author/licence сохранена в snapshots и галерее → происхождение каждого изображения доступно → сохранять поля при экспорте.

## Проверенные варианты (validation)
```
        scorer  ridge_alpha  accuracy  macro_f1  true_accept_rate  external_any_poi_far  hard_wrong_far
gallery_cosine          NaN  0.788079  0.733041          0.615894              0.008726        0.013245
balanced_ridge         0.01  0.801325  0.771078          0.576159              0.008726        0.006623
balanced_ridge         0.10  0.854305  0.841510          0.629139              0.008726        0.000000
balanced_ridge         1.00  0.827815  0.764385          0.629139              0.008726        0.000000
```

Исходники, разбиение и предыдущие модели не изменялись. Автоматического переключения используемой модели нет.
